# 07. Computer-Using Agents: Web & Multimodal

Computer-using agents operate existing interfaces instead of purpose-built APIs. This notebook demonstrates how agents observe screens, ground actions, and interact using different tooling paradigms.

*Note: To run these examples, you will need the `anthropic`, `playwright`, and `browser-use` libraries installed, along with valid API keys.*

## Setup

Install the necessary libraries if you haven't already:

In [ ]:
!pip install anthropic playwright browser-use langchain-openai
!playwright install

## 1. Raw Model API (Anthropic Computer Use)

At the lowest level of abstraction, models are fine-tuned to output specific x/y coordinate clicks and keystrokes based on screenshots you provide. You are responsible for capturing the screen, feeding it to the model, and executing the requested actions using tools like PyAutoGUI.

In [ ]:
import os
import anthropic

# Ensure you have your API key set
# os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'

try:
    client = anthropic.Anthropic()
    
    print('Sending prompt to Claude to use the computer tool...')
    response = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=1024,
        tools=[{
            "type": "computer_20241022",
            "name": "computer",
            "display_width_px": 1024,
            "display_height_px": 768,
            "display_number": 1
        }],
        messages=[{
            "role": "user",
            "content": "Click the 'Submit' button on the screen."
        }]
    )
    
    # Extract tool calls from the response
    for block in response.content:
        if block.type == 'tool_use' and block.name == 'computer':
            print("Model requested action:", block.input)
except Exception as e:
    print(f"API Error (did you set your key?): {e}")

## 2. Traditional DOM Automation (Playwright)

For decades, we interacted with web apps deterministically using DOM selectors. This is highly reliable but brittle: if a developer changes the `id` of a button, the script breaks entirely.

In [ ]:
from playwright.sync_api import sync_playwright

def run_playwright():
    print('Launching Playwright...')
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto("https://example.com")
        
        try:
            # Attempting to click a highly specific CSS selector that doesn't exist
            print('Attempting to click button#login-submit...')
            page.click("button#login-submit", timeout=2000)
        except Exception as e:
            print(f'\nCRITICAL FAILURE:\n{e}')
            print('\nBecause Playwright relies on exact selectors, it cannot recover if the UI drifts.')
        finally:
            browser.close()

run_playwright()

## 3. Agentic Browser Wrappers (Browser-Use)

Modern frameworks abstract the browser entirely. Instead of writing DOM selectors, you provide a high-level natural language instruction. The library extracts the DOM/Accessibility Tree (AXTree), sends it to an LLM, and executes the chosen elements natively.

In [ ]:
import asyncio
from browser_use import Agent
from langchain_openai import ChatOpenAI

# Ensure you have your API key set
# os.environ['OPENAI_API_KEY'] = 'your-key-here'

async def run_browser_agent():
    try:
        print('Initializing Browser-Use Agent...')
        agent = Agent(
            task="Go to example.com and extract the main header text.",
            llm=ChatOpenAI(model="gpt-4o")
        )
        result = await agent.run()
        print('\nAgent execution finished! Result:')
        print(result)
    except Exception as e:
        print(f"API Error (did you set your key?): {e}")

# In Jupyter, we can await directly. In a standard script, use asyncio.run()
await run_browser_agent()